# GenQuiz — Supervised Fine-Tuning (SFT) & QLoRA Pipeline
### Fine-Tuning an Open-Source LLM for Structured, Grounded Quiz Generation

This Google Colab notebook trains a LoRA adapter on an open-source instruction model (`Qwen/Qwen2.5-1.5B-Instruct`) using **QLoRA (4-bit quantization)** and Hugging Face `trl.SFTTrainer`.

#### System Overview
- **Base Model**: `Qwen/Qwen2.5-1.5B-Instruct` (Un-gated, no Hugging Face token required, state-of-the-art JSON reasoning)
- **Training Method**: 4-bit QLoRA with `bitsandbytes`, `peft`, and `trl`
- **Hardware Requirements**: Free Google Colab T4 GPU (uses ~2.2 GB VRAM for training; fits easily within 16 GB VRAM)
- **Training Time**: ~3–5 minutes for 5 epochs on 40 training examples
- **Output**: Exportable LoRA adapter directory `genquiz_lora_adapter/` compatible with local CPU/GPU Streamlit deployment in GenQuiz

## Step 1: Install Required Libraries
Run this cell to install `transformers`, `peft`, `accelerate`, `bitsandbytes`, `trl`, and `datasets`.

In [1]:
# Install dependencies (optimized for Google Colab)
!pip install -q -U transformers peft accelerate bitsandbytes trl datasets

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: No GPU detected! Please change runtime type to T4 GPU (Runtime -> Change runtime type -> T4 GPU).")


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


PyTorch Version: 2.14.1+cu126
CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.00 GB


## Step 2: Upload or Create Training and Validation Data
Upload the `train_sft.jsonl` and `val_sft.jsonl` files prepared by `prepare_dataset.py` from your GenQuiz project.
If you are running standalone, the cell below provides a fallback uploader or generates the stratified dataset directly.

In [2]:

import os
import json
import shutil
from pathlib import Path

# Your GenQuiz project directory
PROJECT_DIR = Path(r"D:\GenQuiz")

data_dir = PROJECT_DIR / "data"
data_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "train_sft.jsonl"
val_path = data_dir / "val_sft.jsonl"

# Check whether the dataset files exist
missing = [
    name for name, path in [
        ("train_sft.jsonl", train_path),
        ("val_sft.jsonl", val_path),
    ]
    if not path.exists()
]

if missing:
    print("Dataset files not found:", ", ".join(missing))
    print("Place both files in:", data_dir)
else:
    # Validate that each non-empty line is valid JSON
    for path in (train_path, val_path):
        with path.open("r", encoding="utf-8") as f:
            records = [json.loads(line) for line in f if line.strip()]
        print(f"{path.name}: {len(records)} samples")

    print("\nDataset verification complete.")


train_sft.jsonl: 40 samples
val_sft.jsonl: 10 samples

Dataset verification complete.


## Step 3: Load the Base Model and Tokenizer with 4-bit QLoRA
We load `Qwen/Qwen2.5-1.5B-Instruct` using `bitsandbytes` NF4 quantization.
This reduces model memory from ~3 GB to ~950 MB in VRAM.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading base model: {BASE_MODEL_NAME}...")

# 4-Bit Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Base model and tokenizer successfully loaded into VRAM.")

d:\AI-Training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading base model: Qwen/Qwen2.5-1.5B-Instruct...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1023 21:54:43.263000 26684 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 338/338 [00:03<00:00, 84.90it/s] 


Base model and tokenizer successfully loaded into VRAM.


## Step 4: Configure LoRA Adapters (PEFT)
We configure low-rank adapters targeting all linear attention and MLP projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
This ensures maximum expressiveness while training only ~1.5% of total parameters.

In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Keep trainable adapter weights compatible with the FP16 GradScaler.
for parameter in model.parameters():
    if parameter.requires_grad and parameter.dtype != torch.float16:
        parameter.data = parameter.data.to(torch.float16)

model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## Step 5: Prepare Datasets for SFTTrainer
We load the conversational JSONL files using Hugging Face `datasets` and apply the ChatML chat template.

In [5]:

from datasets import load_dataset

# Convert pathlib paths to strings for compatibility
dataset = load_dataset(
    "json",
    data_files={
        "train": str(train_path),
        "validation": str(val_path),
    }
)

def apply_chat_formatting(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

formatted_dataset = dataset.map(apply_chat_formatting)

print("Sample formatted training example:")
print(formatted_dataset["train"][0]["text"][:600] + "...")


Sample formatted training example:
<|im_start|>system
You are an expert assessment generator. Based strictly on the provided context passage and instructions, generate a pedagogically aligned evaluation question with its answer, explanation, and metadata. Output valid JSON only conforming to the required schema.<|im_end|>
<|im_start|>user
SOURCE CONTEXT:
A pretrained base model is good at continuing text but may not follow user requests reliably. Further training on prompt-response pairs, and sometimes human preference feedback, changes this behavior.

SPECIFICATIONS:
- Topic: Large Language Models (LLMs)
- Question Type: MCQ
-...


## Step 6: Initialize and Run SFTTrainer
We use Hugging Face TRL's `SFTTrainer` with a cosine learning rate scheduler, evaluation at each epoch, and FP16 mixed-precision.

In [6]:

from trl import SFTTrainer, SFTConfig
import torch

training_args = SFTConfig(
    output_dir="./genquiz_checkpoints",
    dataset_text_field="text",
    max_length=1024,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    gradient_checkpointing=True,

    # Use native BF16 AMP; FP16 GradScaler cannot unscale BF16 gradients.
    fp16=False,
    bf16=True,

    per_device_eval_batch_size=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=5,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    args=training_args,
)

import torch

print("Trainer FP16:", trainer.args.fp16)
print("Trainer BF16:", trainer.args.bf16)
print("Model parameter dtypes:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

print("GPU supports BF16:", torch.cuda.is_bf16_supported())


print("Starting fine-tuning...")
train_result = trainer.train()

print("Fine-tuning complete!")
print(train_result.metrics)


Trainer FP16: False
Trainer BF16: True
Model parameter dtypes:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.bfloat16
GPU supports BF16: True
Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.090172,1.624237,1.931675,0.664352,13163.000000
2,0.883884,0.958620,0.931181,0.799940,26326.000000
3,0.818173,0.919021,0.814636,0.800488,39489.000000
4,0.661833,0.916125,0.750848,0.804712,52652.000000
5,0.643387,0.919460,0.734139,0.803955,65815.000000


Fine-tuning complete!
{'train_runtime': 127.5216, 'train_samples_per_second': 1.568, 'train_steps_per_second': 0.392, 'total_flos': 524731954928640.0, 'train_loss': 1.1309122562408447, 'epoch': 5.0}


## Step 7: Save the Trained LoRA Adapter
We save the LoRA adapter weights and tokenizer configuration into `./genquiz_lora_adapter`.

In [7]:
ADAPTER_OUTPUT_DIR = "genquiz_lora_adapter"
os.makedirs(ADAPTER_OUTPUT_DIR, exist_ok=True)

# Save LoRA adapter weights and config
trainer.model.save_pretrained(ADAPTER_OUTPUT_DIR)
tokenizer.save_pretrained(ADAPTER_OUTPUT_DIR)

# Create adapter metadata README
readme_content = f"""---
base_model: {BASE_MODEL_NAME}
library_name: peft
pipeline_tag: text-generation
tags:
- genquiz
- education
- assessment-generation
- qlora
---

# GenQuiz LoRA Adapter

This adapter was fine-tuned on `{BASE_MODEL_NAME}` using QLoRA for structured, document-grounded quiz generation.
Compatible with the GenQuiz local model inference engine.
"""

with open(os.path.join(ADAPTER_OUTPUT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"Adapter files saved to {ADAPTER_OUTPUT_DIR}:")
for f in os.listdir(ADAPTER_OUTPUT_DIR):
    print(f"  - {f}")

Adapter files saved to genquiz_lora_adapter:
  - adapter_config.json
  - adapter_model.safetensors
  - chat_template.jinja
  - README.md
  - tokenizer.json
  - tokenizer_config.json


## Step 8: Test Inference with the Fine-Tuned Model
We test generating a grounded question from an unseen context passage.

In [8]:
test_context = (
    "Multi-Head Attention allows the Transformer to jointly attend to information "
    "from different representation subspaces at different positions. Instead of performing "
    "a single attention function with d_model-dimensional queries, keys, and values, "
    "it linearly projects queries, keys, and values h times with different learned linear projections."
)

test_messages = [
    {
        "role": "system",
        "content": "You are an expert assessment generator. Based strictly on the provided context passage and instructions, generate a pedagogically aligned evaluation question with its answer, explanation, and metadata. Output valid JSON only conforming to the required schema."
    },
    {
        "role": "user",
        "content": (
            f"SOURCE CONTEXT:\n{test_context}\n\n"
            "SPECIFICATIONS:\n"
            "- Topic: Multi-Head Attention\n"
            "- Question Type: MCQ\n"
            "- Difficulty Level: Medium\n"
            "- Bloom's Taxonomy Level: Understand\n\n"
            "Generate one Medium-level MCQ question strictly grounded in the source context above. Include the correct answer, educational explanation, and options (if MCQ). Return valid JSON only."
        )
    }
]

# Format with chat template
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
raw_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print("=== RAW MODEL GENERATION ===")
print(raw_response)

# Parse JSON
try:
    clean_json = raw_response.strip()
    if clean_json.startswith("```"):
        import re
        clean_json = re.sub(r"^```[a-zA-Z0-9_-]*\n?", "", clean_json)
        clean_json = re.sub(r"```$", "", clean_json).strip()
    parsed = json.loads(clean_json)
    print("\nJSON Schema Validation: SUCCESS!")
    print(f"Question: {parsed.get('question')}")
    print(f"Answer: {parsed.get('correct_answer')}")
    print(f"Explanation: {parsed.get('explanation')}")
except Exception as e:
    print(f"JSON parsing check: {e}")

=== RAW MODEL GENERATION ===
{
  "question_id": "GAI026",
  "question": "A model that uses Multi-Head Attention has three heads. What is the maximum number of distinct parameters this model will have compared to a regular self-attention model?",
  "question_type": "MCQ",
  "options": {
    "A": "1d + 3h(d+q) + 3h(d+k) + 3h(d+v)",
    "B": "4d + 9h(d+q) + 9h(d+k) + 9h(d+v)",
    "C": "8d + 15h(d+q) + 15h(d+k) + 15h(d+v)",
    "D": "16d + 21h(d+q) + 21h(d+k) + 21h(d+v)"
  },
  "correct_answer": "C",
  "explanation": "Each head requires its own query, key, value projection matrices plus biases.",
  "topic": "Multi-Head Attention",
  "difficulty": "Medium",
  "bloom_level": "Understand"
}

JSON Schema Validation: SUCCESS!
Question: A model that uses Multi-Head Attention has three heads. What is the maximum number of distinct parameters this model will have compared to a regular self-attention model?
Answer: C
Explanation: Each head requires its own query, key, value projection matrices plu

## Step 9: Package and Download the Adapter for GenQuiz
Run this cell to zip the `genquiz_lora_adapter` directory and download it directly to your computer.

### How to use in GenQuiz:
1. Extract `genquiz_lora_adapter.zip` into your local `D:\GenQuiz` project directory as `fine_tuned_adapter` (or `models/genquiz_lora_adapter`).
2. Start GenQuiz (`streamlit run app.py`).
3. Select **'Fine-Tuned Local Model (QLoRA)'** from the Target LLM Provider dropdown in the sidebar.
4. The system will load the local weights and generate quiz questions without requiring any external API keys!

In [10]:
import shutil

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

# Zip adapter folder
zip_filename = "genquiz_lora_adapter.zip"
shutil.make_archive("genquiz_lora_adapter", "zip", ADAPTER_OUTPUT_DIR)
print(f"Created {zip_filename} ({os.path.getsize(zip_filename) / (1024*1024):.2f} MB)")

# Trigger automatic download in Colab; local notebooks can open the ZIP directly.
if files is not None:
    print("Downloading adapter to local machine...")
    files.download(zip_filename)
else:
    print(f"Adapter ZIP is ready at: {os.path.abspath(zip_filename)}")

Created genquiz_lora_adapter.zip (29.69 MB)
Adapter ZIP is ready at: d:\GenQuiz\genquiz_lora_adapter.zip
